# Claude를 위한 커스텀 스킬 만들기

조직의 전문 지식과 업무 흐름으로 Claude의 역량을 확장하는 커스텀 스킬을 만들고, 배포하고, 관리하는 방법을 배웁니다.

## 목차

1. [들어가며와 준비](#introduction)
2. [커스텀 스킬 구조 이해하기](#architecture)
3. [예제 1: 재무 비율 계산기](#financial-ratio)
4. [예제 2: 회사 브랜드 가이드라인](#brand-guidelines)
5. [예제 3: 재무 모델링 스위트](#financial-modeling)
6. [스킬 관리와 버전 관리](#management)
7. [모범 사례와 프로덕션 팁](#best-practices)
8. [문제 해결](#troubleshooting)

## 1. 들어가며와 준비 {#introduction}

### 커스텀 스킬이란?

**커스텀 스킬**은 조직 고유의 업무 흐름, 도메인 지식, 모범 사례를 Claude에 가르치기 위해 여러분이 만드는 전문성 패키지입니다. Anthropic이 미리 만들어 둔 스킬(Excel, PowerPoint, PDF)과 달리, 커스텀 스킬로는 다음이 가능합니다.

- **조직 지식 성문화** — 팀 고유의 방법론을 담아냅니다
- **일관성 보장** — 모든 상호작용에 같은 기준을 적용합니다
- **복잡한 워크플로 자동화** — 여러 단계를 이어 붙입니다
- **지식 재산 보호** — 독자적인 방법을 안전하게 유지합니다

### 주요 이점

| 이점 | 설명 |
|---------|-------------|
| **규모 있는 전문성** | 모든 Claude 상호작용에 전문 지식을 적용 |
| **버전 관리** | 변경을 추적하고 필요하면 되돌리기 |
| **조합 가능성** | 복잡한 작업을 위해 여러 스킬을 결합 |
| **비공개성** | 스킬은 조직 내부에만 남습니다 |

### 사전 준비

시작하기 전에 다음을 확인하세요.
- [노트북 1: 스킬 소개](01_skills_introduction.ipynb) 완료
- Skills 베타에 접근할 수 있는 Anthropic API 키
- 로컬 SDK가 설치된 Python 환경

### 환경 준비

환경을 설정하고 필요한 라이브러리를 가져오겠습니다:

In [ ]:
import os
import sys
from pathlib import Path
from typing import Any

# Add parent directory for imports
sys.path.insert(0, str(Path.cwd().parent))

from anthropic import Anthropic
from anthropic.lib import files_from_dir
from dotenv import load_dotenv

# Import our utilities
from file_utils import (
    download_all_files,
    extract_file_ids,
    print_download_summary,
)

# We'll create skill_utils later in this notebook
# from skill_utils import (
#     create_skill,
#     list_skills,
#     delete_skill,
#     test_skill
# )

# Load environment variables
load_dotenv(Path.cwd().parent / ".env")

API_KEY = os.getenv("ANTHROPIC_API_KEY")
MODEL = os.getenv("ANTHROPIC_MODEL", "claude-sonnet-4-6")

if not API_KEY:
    raise ValueError(
        "ANTHROPIC_API_KEY not found. Copy ../.env.example to ../.env and add your API key."
    )

# Initialize client with Skills beta
client = Anthropic(api_key=API_KEY, default_headers={"anthropic-beta": "skills-2025-10-02"})

# Setup directories
SKILLS_DIR = Path.cwd().parent / "custom_skills"
OUTPUT_DIR = Path.cwd().parent / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

print("✓ API key loaded")
print(f"✓ Using model: {MODEL}")
print(f"✓ Custom skills directory: {SKILLS_DIR}")
print(f"✓ Output directory: {OUTPUT_DIR}")
print("\n📝 Skills beta header configured for skill management")

## 2. 커스텀 스킬 구조 이해하기 {#architecture}

### 스킬 구조

모든 커스텀 스킬은 다음 디렉터리 구조를 따릅니다.

```
skill_name/
├── SKILL.md           # REQUIRED: Instructions with YAML frontmatter
├── *.md               # Optional: Any additional .md files (documentation, guides)
├── scripts/           # Optional: Executable code
│   ├── process.py
│   └── utils.js
└── resources/         # Optional: Templates, data files
    └── template.xlsx
```

**중요:** 
- **SKILL.md만이 유일한 필수 파일입니다** — 나머지는 모두 선택 사항입니다
- **여러 .md 파일 허용** — 최상위 폴더에 마크다운 파일을 원하는 만큼 둘 수 있습니다
- **모든 .md 파일이 적재됩니다** — SKILL.md와 REFERENCE.md만이 아니라 여러분이 포함한 모든 .md 파일이 적재됩니다
- **필요에 맞게 정리하세요** — 복잡한 문서는 여러 .md 파일로 구조화하세요

📖 엔지니어링 블로그 글 [스킬로 에이전트를 현실 세계에 대비시키기](https://www.anthropic.com/engineering/equipping-agents-for-the-real-world-with-agent-skills)도 읽어 보세요

### 스킬은 마크다운만이 아닙니다

![스킬에는 스크립트와 파일을 포함할 수 있습니다](../assets/not-just-markdown.png)

스킬에는 다양한 파일 유형을 묶을 수 있습니다.
- **마크다운 파일**: 지시와 문서(SKILL.md, REFERENCE.md 등)
- **스크립트**: 복잡한 작업을 위한 Python, JavaScript 등의 실행 코드
- **템플릿**: 커스터마이즈할 수 있는 미리 만들어진 파일(Excel 템플릿, 문서 템플릿)
- **리소스**: 보조 데이터 파일, 설정, 자산

### SKILL.md 요구 사항

`SKILL.md` 파일에는 다음이 포함되어야 합니다.

1. **YAML 프런트매터**(name: 64자, description: 1024자)
   - `name`: 하이픈을 포함한 소문자 영숫자(필수)
   - `description`: 스킬이 하는 일에 대한 간단한 설명(필수)

2. **지시문**(마크다운 형식)
   - Claude를 위한 명확한 안내
   - 사용 예시
   - 제약이나 규칙
   - 권장: 5,000 토큰 이하로 유지

### 추가 문서 파일

더 나은 정리를 위해 여러 마크다운 파일을 포함할 수 있습니다.

```
skill_name/
├── SKILL.md           # Main instructions (required)
├── REFERENCE.md       # API reference (optional)
├── EXAMPLES.md        # Usage examples (optional)
├── TROUBLESHOOTING.md # Common issues (optional)
└── CHANGELOG.md       # Version history (optional)
```

루트 디렉터리의 모든 `.md` 파일은 스킬이 적재될 때 Claude가 사용할 수 있습니다.

### 번들 파일 예시

![스킬에 묶인 파일들](../assets/skills-bundled-files.png)

이 예시는 스킬이 여러 파일을 묶는 방법을 보여 줍니다.
- **SKILL.md**: 색상, 타이포그래피, 섹션이 담긴 주요 지시문
- **slide-decks.md**: 특정 사용 사례를 위한 추가 문서
- **스크립트와 리소스**: 스킬 실행 중에 참조하고 사용할 수 있습니다

### 점진적 공개

스킬은 토큰 사용을 최적화하기 위해 세 단계로 적재됩니다.

| 단계 | 내용 | 토큰 비용 | 적재 시점 |
|-------|---------|------------|-------------|
| **1. 메타데이터** | 이름과 설명 | name: 64자, description: 1024자 | 항상 보임 |
| **2. 지시문** | 모든 .md 파일 | 5,000 토큰 미만 권장 | 관련이 있을 때 |
| **3. 리소스** | 스크립트와 파일 | 필요한 만큼 | 실행 중 |

### API 워크플로

```python
# 1. Create skill
skill = client.beta.skills.create(
    display_title="My Skill",
    files=files_from_dir("path/to/skill")
)

# 2. Use in messages
response = client.beta.messages.create(
    container={
        "skills": [{
            "type": "custom",
            "skill_id": skill.id,
            "version": "latest"
        }]
    },
    # ... rest of message parameters
)
```

### 모범 사례

스킬 제작과 모범 사례에 대한 자세한 안내는 다음을 참고하세요.
- [Claude 스킬 모범 사례](https://docs.claude.com/en/docs/agents-and-tools/agent-skills/best-practices)
- [스킬 문서](https://docs.claude.com/en/docs/agents-and-tools/agent-skills/overview)

### 스킬 유틸리티 함수 만들기

스킬 관리를 위한 헬퍼 함수를 만들어 보겠습니다:

In [ ]:
def create_skill(client: Anthropic, skill_path: str, display_title: str) -> dict[str, Any]:
    """
    Create a new custom skill from a directory.

    Args:
        client: Anthropic client instance
        skill_path: Path to skill directory
        display_title: Human-readable skill name

    Returns:
        Dictionary with skill_id, version, and metadata
    """
    try:
        # Create skill using files_from_dir
        skill = client.beta.skills.create(
            display_title=display_title, files=files_from_dir(skill_path)
        )

        return {
            "success": True,
            "skill_id": skill.id,
            "display_title": skill.display_title,
            "latest_version": skill.latest_version,
            "created_at": skill.created_at,
            "source": skill.source,
        }
    except Exception as e:
        return {"success": False, "error": str(e)}


def list_custom_skills(client: Anthropic) -> list[dict[str, Any]]:
    """
    List all custom skills in the workspace.

    Returns:
        List of skill dictionaries
    """
    try:
        skills_response = client.beta.skills.list(source="custom")

        skills = []
        for skill in skills_response.data:
            skills.append(
                {
                    "skill_id": skill.id,
                    "display_title": skill.display_title,
                    "latest_version": skill.latest_version,
                    "created_at": skill.created_at,
                    "updated_at": skill.updated_at,
                }
            )

        return skills
    except Exception as e:
        print(f"Error listing skills: {e}")
        return []


def delete_skill(client: Anthropic, skill_id: str) -> bool:
    """
    Delete a custom skill and all its versions.

    Args:
        client: Anthropic client
        skill_id: ID of skill to delete

    Returns:
        True if successful, False otherwise
    """
    try:
        # First delete all versions
        versions = client.beta.skills.versions.list(skill_id=skill_id)

        for version in versions.data:
            client.beta.skills.versions.delete(skill_id=skill_id, version=version.version)

        # Then delete the skill itself
        client.beta.skills.delete(skill_id)
        return True

    except Exception as e:
        print(f"Error deleting skill: {e}")
        return False


def test_skill(
    client: Anthropic,
    skill_id: str,
    test_prompt: str,
    model: str = "claude-sonnet-4-6",
) -> Any:
    """
    Test a custom skill with a prompt.

    Args:
        client: Anthropic client
        skill_id: ID of skill to test
        test_prompt: Prompt to test the skill
        model: Model to use for testing

    Returns:
        Response from Claude
    """
    response = client.beta.messages.create(
        model=model,
        max_tokens=4096,
        container={"skills": [{"type": "custom", "skill_id": skill_id, "version": "latest"}]},
        tools=[{"type": "code_execution_20250825", "name": "code_execution"}],
        messages=[{"role": "user", "content": test_prompt}],
        betas=[
            "code-execution-2025-08-25",
            "files-api-2025-04-14",
            "skills-2025-10-02",
        ],
    )

    return response


print("✓ Skill utility functions defined")
print("  - create_skill()")
print("  - list_custom_skills()")
print("  - delete_skill()")
print("  - test_skill()")

### 기존 커스텀 스킬 확인하기

워크스페이스에 이미 있는 커스텀 스킬이 있는지 살펴보겠습니다:

### ⚠️ 중요: 시작 전에 기존 스킬 정리하기

이 노트북을 다시 실행하는 경우 이전 세션의 스킬이 남아 있을 수 있습니다. 스킬은 표시 제목이 중복될 수 없으므로 세 가지 선택지가 있습니다.

1. **기존 스킬 삭제**(테스트에는 권장) — 깨끗한 상태에서 시작
2. **다른 표시 제목 사용** — 이름에 타임스탬프나 버전 번호 추가
3. **기존 스킬을 새 버전으로 갱신** — [스킬 관리와 버전 관리](#management) 절 참고

기존 스킬을 확인하고 필요하면 정리해 보겠습니다:

In [ ]:
# Check for existing skills that might conflict
existing_skills = list_custom_skills(client)
skill_titles_to_create = [
    "Financial Ratio Analyzer",
    "Corporate Brand Guidelines",
    "Financial Modeling Suite",
]
conflicting_skills = []

if existing_skills:
    print(f"Found {len(existing_skills)} existing custom skill(s):")
    for skill in existing_skills:
        print(f"  - {skill['display_title']} (ID: {skill['skill_id']})")
        if skill["display_title"] in skill_titles_to_create:
            conflicting_skills.append(skill)

    if conflicting_skills:
        print(
            f"\n⚠️ Found {len(conflicting_skills)} skill(s) that will conflict with this notebook:"
        )
        for skill in conflicting_skills:
            print(f"  - {skill['display_title']} (ID: {skill['skill_id']})")

        print("\n" + "=" * 70)
        print("To clean up these skills and start fresh, uncomment and run:")
        print("=" * 70)
        print("\n# UNCOMMENT THE LINES BELOW TO DELETE CONFLICTING SKILLS:")
        print("# for skill in conflicting_skills:")
        print("#     if delete_skill(client, skill['skill_id']):")
        print("#         print(f\"✅ Deleted: {skill['display_title']}\")")
        print("#     else:")
        print("#         print(f\"❌ Failed to delete: {skill['display_title']}\")")

        # for skill in conflicting_skills:
        #     if delete_skill(client, skill['skill_id']):
        #         print(f"✅ Deleted: {skill['display_title']}")
        #     else:
        #         print(f"❌ Failed to delete: {skill['display_title']}")
    else:
        print("\n✅ No conflicting skills found. Ready to proceed!")
else:
    print("✅ No existing custom skills found. Ready to create new ones!")

## 3. 예제 1: 재무 비율 계산기 {#financial-ratio}

첫 커스텀 스킬을 만들어 보겠습니다. 기업의 재무 건전성을 분석할 수 있는 재무 비율 계산기입니다.

### 스킬 개요

**재무 비율 계산기** 스킬은 다음을 수행합니다.
- 주요 재무 비율 계산(ROE, P/E, 유동비율 등)
- 업종 맥락을 반영한 비율 해석
- 서식이 적용된 보고서 생성
- 여러 데이터 형식(CSV, JSON, 텍스트) 지원

### 재무 분석기 스킬 업로드하기

이제 재무 분석기 스킬을 Claude에 업로드하겠습니다:

In [ ]:
# Upload the Financial Analyzer skill
financial_skill_path = SKILLS_DIR / "analyzing-financial-statements"

if financial_skill_path.exists():
    print("Uploading Financial Analyzer skill...")
    result = create_skill(client, str(financial_skill_path), "Financial Ratio Analyzer")

    if result["success"]:
        financial_skill_id = result["skill_id"]
        print("✅ Skill uploaded successfully!")
        print(f"   Skill ID: {financial_skill_id}")
        print(f"   Version: {result['latest_version']}")
        print(f"   Created: {result['created_at']}")
    else:
        print(f"❌ Upload failed: {result['error']}")
        if "cannot reuse an existing display_title" in str(result["error"]):
            print("\n💡 Solution: A skill with this name already exists.")
            print("   Run the 'Clean Up Existing Skills' cell above to delete it first,")
            print("   or change the display_title to something unique.")
else:
    print(f"⚠️ Skill directory not found: {financial_skill_path}")
    print(
        "Please ensure the custom_skills directory contains the analyzing-financial-statements folder."
    )

### 재무 분석기 스킬 테스트하기

샘플 재무 데이터로 스킬을 시험해 보겠습니다:

In [ ]:
# Test the Financial Analyzer skill
if "financial_skill_id" in locals():
    test_prompt = """
    Calculate financial ratios for this company:

    Income Statement:
    - Revenue: $1,000M
    - EBITDA: $200M
    - Net Income: $120M

    Balance Sheet:
    - Total Assets: $2,000M
    - Current Assets: $500M
    - Current Liabilities: $300M
    - Total Debt: $400M
    - Shareholders Equity: $1,200M

    Market Data:
    - Share Price: $50
    - Shares Outstanding: 100M

    Please calculate key ratios and provide analysis.
    """

    print("Testing Financial Analyzer skill...")
    response = test_skill(client, financial_skill_id, test_prompt)

    # Print response
    for content in response.content:
        if content.type == "text":
            print(content.text)
else:
    print("⚠️ Please upload the Financial Analyzer skill first (run the previous cell)")

## 4. 예제 2: 회사 브랜드 가이드라인 {#brand-guidelines}

이번에는 모든 문서가 회사의 브랜드 기준을 따르게 하는 스킬을 만들어 보겠습니다.

### 스킬 개요

**브랜드 가이드라인** 스킬은 다음을 수행합니다.
- 일관된 색상, 글꼴, 레이아웃 적용
- 로고 배치와 사용 규칙 준수
- 전문적인 어조와 메시지 유지
- 모든 문서 유형(Excel, PowerPoint, PDF)에서 동작

In [ ]:
# Upload the Brand Guidelines skill
brand_skill_path = SKILLS_DIR / "applying-brand-guidelines"

if brand_skill_path.exists():
    print("Uploading Brand Guidelines skill...")
    result = create_skill(client, str(brand_skill_path), "Corporate Brand Guidelines")

    if result["success"]:
        brand_skill_id = result["skill_id"]
        print("✅ Skill uploaded successfully!")
        print(f"   Skill ID: {brand_skill_id}")
        print(f"   Version: {result['latest_version']}")
    else:
        print(f"❌ Upload failed: {result['error']}")
        if "cannot reuse an existing display_title" in str(result["error"]):
            print("\n💡 Solution: A skill with this name already exists.")
            print("   Run the 'Clean Up Existing Skills' cell above to delete it first,")
            print("   or change the display_title to something unique.")
else:
    print(f"⚠️ Skill directory not found: {brand_skill_path}")

### 문서 생성으로 브랜드 가이드라인 테스트하기

브랜드가 적용된 PowerPoint 발표 자료를 만들어 브랜드 스킬을 시험해 보겠습니다:

In [ ]:
# Test Brand Guidelines skill with PowerPoint creation
if "brand_skill_id" in locals():
    # Combine brand skill with Anthropic's pptx skill
    response = client.beta.messages.create(
        model=MODEL,
        max_tokens=4096,
        container={
            "skills": [
                {"type": "custom", "skill_id": brand_skill_id, "version": "latest"},
                {"type": "anthropic", "skill_id": "pptx", "version": "latest"},
            ]
        },
        tools=[{"type": "code_execution_20250825", "name": "code_execution"}],
        messages=[
            {
                "role": "user",
                "content": """Create a 3-slide PowerPoint presentation following Acme Corporation brand guidelines:

            Slide 1: Title slide for "Q4 2025 Results"
            Slide 2: Revenue Overview with a chart showing Q1-Q4 growth
            Slide 3: Key Achievements (3 bullet points)

            Apply all brand colors, fonts, and formatting standards.
            """,
            }
        ],
        betas=[
            "code-execution-2025-08-25",
            "files-api-2025-04-14",
            "skills-2025-10-02",
        ],
    )

    print("Response from Claude:")
    for content in response.content:
        if content.type == "text":
            print(content.text[:500] + "..." if len(content.text) > 500 else content.text)

    # Download generated file
    file_ids = extract_file_ids(response)
    if file_ids:
        results = download_all_files(
            client, response, output_dir=str(OUTPUT_DIR), prefix="branded_"
        )
        print_download_summary(results)
else:
    print("⚠️ Please upload the Brand Guidelines skill first")

## 5. 예제 3: 재무 모델링 스위트 {#financial-modeling}

가장 고급 스킬을 만들어 보겠습니다. 가치 평가와 위험 분석을 위한 종합 재무 모델링 스위트입니다.

### 스킬 개요

**재무 모델링 스위트** 스킬은 다음을 제공합니다.
- **DCF 가치 평가**: 완전한 현금흐름할인 모델
- **민감도 분석**: 변수가 가치 평가에 미치는 영향 시험
- **몬테카를로 시뮬레이션**: 확률 분포를 활용한 위험 모델링
- **시나리오 계획**: 최선/기본/최악 사례 분석

복잡한 계산과 실무 수준의 재무 모델링을 담은 다중 파일 스킬을 보여 줍니다.

### 재무 모델링 스위트 업로드하기

먼저 재무 모델링 스킬을 업로드합니다:

In [ ]:
# Upload the Financial Modeling Suite skill
modeling_skill_path = SKILLS_DIR / "creating-financial-models"

if modeling_skill_path.exists():
    print("Uploading Financial Modeling Suite skill...")
    result = create_skill(client, str(modeling_skill_path), "Financial Modeling Suite")

    if result["success"]:
        modeling_skill_id = result["skill_id"]
        print("✅ Skill uploaded successfully!")
        print(f"   Skill ID: {modeling_skill_id}")
        print(f"   Version: {result['latest_version']}")
        print("\nThis skill includes:")
        print("   - DCF valuation model (dcf_model.py)")
        print("   - Sensitivity analysis framework (sensitivity_analysis.py)")
        print("   - Monte Carlo simulation capabilities")
        print("   - Scenario planning tools")
    else:
        print(f"❌ Upload failed: {result['error']}")
else:
    print(f"⚠️ Skill directory not found: {modeling_skill_path}")
    print(
        "Please ensure the custom_skills directory contains the creating-financial-models folder."
    )

### 재무 모델링 스위트 테스트하기

DCF 가치 평가 요청으로 고급 모델링 기능을 시험해 보겠습니다:

In [ ]:
# Test the Financial Modeling Suite with a DCF valuation
if "modeling_skill_id" in locals():
    dcf_test_prompt = """
    Perform a DCF valuation for TechCorp with the following data:

    Historical Financials (Last 3 Years):
    - Revenue: $500M, $600M, $750M
    - EBITDA Margin: 25%, 27%, 30%
    - CapEx: $50M, $55M, $60M
    - Working Capital: 15% of revenue

    Projections:
    - Revenue growth: 20% for years 1-3, then declining to 5% by year 5
    - EBITDA margin expanding to 35% by year 5
    - Terminal growth rate: 3%

    Market Assumptions:
    - WACC: 10%
    - Tax rate: 25%
    - Current net debt: $200M
    - Shares outstanding: 100M

    Please create a complete DCF model with sensitivity analysis on WACC and terminal growth.
    Generate an Excel file with the full model including:
    1. Revenue projections
    2. Free cash flow calculations
    3. Terminal value
    4. Enterprise value to equity value bridge
    5. Sensitivity table
    """

    print("Testing Financial Modeling Suite with DCF valuation...")
    print("=" * 70)
    print("\n⏱️ Note: Complex financial model generation may take 1-2 minutes.\n")

    response = client.beta.messages.create(
        model=MODEL,
        max_tokens=4096,
        container={
            "skills": [
                {"type": "custom", "skill_id": modeling_skill_id, "version": "latest"},
                {"type": "anthropic", "skill_id": "xlsx", "version": "latest"},
            ]
        },
        tools=[{"type": "code_execution_20250825", "name": "code_execution"}],
        messages=[{"role": "user", "content": dcf_test_prompt}],
        betas=[
            "code-execution-2025-08-25",
            "files-api-2025-04-14",
            "skills-2025-10-02",
        ],
    )

    # Print Claude's response
    for content in response.content:
        if content.type == "text":
            # Print first 800 characters to keep output manageable
            text = content.text
            if len(text) > 800:
                print(text[:800] + "\n\n[... Output truncated for brevity ...]")
            else:
                print(text)

    # Download the DCF model if generated
    file_ids = extract_file_ids(response)
    if file_ids:
        print("\n" + "=" * 70)
        print("Downloading generated DCF model...")
        results = download_all_files(
            client, response, output_dir=str(OUTPUT_DIR), prefix="dcf_model_"
        )
        print_download_summary(results)
        print("\n💡 Open the Excel file to explore the complete DCF valuation model!")
else:
    print("⚠️ Please upload the Financial Modeling Suite skill first (run the previous cell)")

## 6. 스킬 관리와 버전 관리 {#management}

시간이 지나며 스킬을 관리하려면 버전 관리, 갱신, 수명 주기 관리를 이해해야 합니다.

### 스킬 목록 보기

워크스페이스의 모든 커스텀 스킬을 한눈에 확인합니다:

In [ ]:
# List all your custom skills
my_skills = list_custom_skills(client)

if my_skills:
    print(f"You have {len(my_skills)} custom skill(s):\n")
    print("=" * 70)
    for i, skill in enumerate(my_skills, 1):
        print(f"\n{i}. {skill['display_title']}")
        print(f"   Skill ID: {skill['skill_id']}")
        print(f"   Current Version: {skill['latest_version']}")
        print(f"   Created: {skill['created_at']}")
        if skill.get("updated_at"):
            print(f"   Last Updated: {skill['updated_at']}")
    print("\n" + "=" * 70)
else:
    print("No custom skills found in your workspace.")

### 새 버전 만들기

스킬은 이력을 유지하고 되돌릴 수 있도록 버전 관리를 지원합니다. 재무 분석기 스킬을 개선하고 새 버전을 만들어 보겠습니다.

#### 1단계: 재무 분석기 개선하기

스킬의 활용도를 높이기 위해 **헬스케어 업종** 기준치를 추가합니다. 사용자 요구에 따라 스킬의 역량을 넓히는 실제 상황입니다.

In [ ]:
# Add healthcare industry benchmarks to the Financial Analyzer
# This demonstrates a realistic skill enhancement scenario

if "financial_skill_id" in locals():
    # Read the current interpret_ratios.py file
    interpret_file_path = SKILLS_DIR / "analyzing-financial-statements" / "interpret_ratios.py"

    with open(interpret_file_path) as f:
        content = f.read()

    # Add healthcare benchmarks after the 'manufacturing' section
    healthcare_benchmarks = """        },
        'healthcare': {
            'current_ratio': {'excellent': 2.3, 'good': 1.8, 'acceptable': 1.4, 'poor': 1.0},
            'debt_to_equity': {'excellent': 0.3, 'good': 0.6, 'acceptable': 1.0, 'poor': 1.8},
            'roe': {'excellent': 0.22, 'good': 0.16, 'acceptable': 0.11, 'poor': 0.07},
            'gross_margin': {'excellent': 0.65, 'good': 0.45, 'acceptable': 0.30, 'poor': 0.20},
            'pe_ratio': {'undervalued': 18, 'fair': 28, 'growth': 40, 'expensive': 55}
        """

    # Find the position after manufacturing section and before the closing brace
    insert_pos = content.find("        }\n    }")  # Find the end of the BENCHMARKS dict

    if insert_pos != -1:
        # Insert the healthcare benchmarks
        new_content = content[:insert_pos] + healthcare_benchmarks + content[insert_pos:]

        # Save the enhanced file
        with open(interpret_file_path, "w") as f:
            f.write(new_content)

        print("✅ Enhanced Financial Analyzer with healthcare industry benchmarks")
        print("\nChanges made:")
        print("  - Added healthcare industry to BENCHMARKS")
        print("  - Includes specific thresholds for:")
        print("    • Current ratio (liquidity)")
        print("    • Debt-to-equity (leverage)")
        print("    • ROE (profitability)")
        print("    • Gross margin")
        print("    • P/E ratio (valuation)")
        print("\n📝 Now we can create a new version of the skill with this enhancement!")
    else:
        print("⚠️ Could not find the correct position to insert healthcare benchmarks")
        print("The file structure may have changed.")
else:
    print("⚠️ Please upload the Financial Analyzer skill first (run cells in Section 3)")

#### 2단계: 새 버전 만들기

스킬을 개선했으니 이 변경을 기록할 새 버전을 만들어 보겠습니다:

In [ ]:
# Create a new version of the enhanced Financial Analyzer skill
def create_skill_version(client: Anthropic, skill_id: str, skill_path: str):
    """Create a new version of an existing skill."""
    try:
        version = client.beta.skills.versions.create(
            skill_id=skill_id, files=files_from_dir(skill_path)
        )
        return {
            "success": True,
            "version": version.version,
            "created_at": version.created_at,
        }
    except Exception as e:
        return {"success": False, "error": str(e)}


# Create the new version with our healthcare enhancement
if "financial_skill_id" in locals():
    print("Creating new version of Financial Analyzer with healthcare benchmarks...")

    result = create_skill_version(
        client, financial_skill_id, str(SKILLS_DIR / "analyzing-financial-statements")
    )

    if result["success"]:
        print("✅ New version created successfully!")
        print(f"   Version: {result['version']}")
        print(f"   Created: {result['created_at']}")
        print("\n📊 Version History:")
        print("   v1: Original skill with tech, retail, financial, manufacturing")
        print(f"   v{result['version']}: Enhanced with healthcare industry benchmarks")
    else:
        print(f"❌ Version creation failed: {result['error']}")
else:
    print("⚠️ Please run the previous cells to upload the skill and make enhancements first")

#### 3단계: 새 버전 테스트하기

헬스케어 기업을 분석해 개선이 잘 되었는지 확인해 보겠습니다:

In [ ]:
# Test the enhanced skill with healthcare industry data
if "financial_skill_id" in locals():
    healthcare_test_prompt = """
    Analyze this healthcare company using the healthcare industry benchmarks:

    Company: MedTech Solutions (Healthcare Industry)

    Income Statement:
    - Revenue: $800M
    - EBITDA: $320M
    - Net Income: $160M

    Balance Sheet:
    - Total Assets: $1,200M
    - Current Assets: $400M
    - Current Liabilities: $200M
    - Total Debt: $300M
    - Shareholders Equity: $700M

    Market Data:
    - Share Price: $75
    - Shares Outstanding: 50M

    Please calculate key ratios and provide healthcare-specific analysis.
    """

    print("Testing enhanced Financial Analyzer with healthcare company...")
    print("=" * 70)

    response = test_skill(client, financial_skill_id, healthcare_test_prompt, MODEL)

    # Print Claude's analysis
    for content in response.content:
        if content.type == "text":
            # Print first 1000 characters to keep output manageable
            text = content.text
            if len(text) > 1000:
                print(text[:1000] + "\n\n[... Output truncated for brevity ...]")
            else:
                print(text)

    print(
        "\n✅ The skill now recognizes 'healthcare' as an industry and applies specific benchmarks!"
    )
else:
    print("⚠️ Please run the previous cells to create the enhanced version first")

### 정리: 스킬 관리하기

테스트가 끝났거나 워크스페이스를 정리해야 할 때 스킬을 선택적으로 제거할 수 있습니다. 지금까지 만든 것을 살펴보고 정리 방법을 안내하겠습니다:

In [ ]:
# Comprehensive skill cleanup with detailed reporting
def review_and_cleanup_skills(client, dry_run=True):
    """
    Review all skills and optionally clean up the ones created in this notebook.

    Args:
        client: Anthropic client
        dry_run: If True, only show what would be deleted without actually deleting
    """
    # Get all current skills
    all_skills = list_custom_skills(client)

    # Skills we created in this notebook
    notebook_skill_names = [
        "Financial Ratio Analyzer",
        "Corporate Brand Guidelines",
        "Financial Modeling Suite",
    ]

    # Track skills created by this notebook
    notebook_skills = []
    other_skills = []

    for skill in all_skills:
        if skill["display_title"] in notebook_skill_names:
            notebook_skills.append(skill)
        else:
            other_skills.append(skill)

    print("=" * 70)
    print("SKILL INVENTORY REPORT")
    print("=" * 70)

    print(f"\nTotal custom skills in workspace: {len(all_skills)}")

    if notebook_skills:
        print(f"\n📚 Skills created by this notebook ({len(notebook_skills)}):")
        for skill in notebook_skills:
            print(f"   • {skill['display_title']}")
            print(f"     ID: {skill['skill_id']}")
            print(f"     Version: {skill['latest_version']}")
            print(f"     Created: {skill['created_at']}")
    else:
        print("\n✅ No skills from this notebook found")

    if other_skills:
        print(f"\n🔧 Other skills in workspace ({len(other_skills)}):")
        for skill in other_skills:
            print(f"   • {skill['display_title']} (v{skill['latest_version']})")

    # Cleanup options
    if notebook_skills:
        print("\n" + "=" * 70)
        print("CLEANUP OPTIONS")
        print("=" * 70)

        if dry_run:
            print("\n🔍 DRY RUN MODE - No skills will be deleted")
            print("\nTo delete the notebook skills, uncomment and run:")
            print("-" * 40)
            print("# review_and_cleanup_skills(client, dry_run=False)")
            print("-" * 40)

            print("\nThis would delete:")
            for skill in notebook_skills:
                print(f"   • {skill['display_title']}")
        else:
            print("\n⚠️ DELETION MODE - Skills will be permanently removed")
            print("\nDeleting notebook skills...")

            success_count = 0
            for skill in notebook_skills:
                if delete_skill(client, skill["skill_id"]):
                    print(f"   ✅ Deleted: {skill['display_title']}")
                    success_count += 1
                else:
                    print(f"   ❌ Failed to delete: {skill['display_title']}")

            print(f"\n📊 Cleanup complete: {success_count}/{len(notebook_skills)} skills deleted")

    return {
        "total_skills": len(all_skills),
        "notebook_skills": len(notebook_skills),
        "other_skills": len(other_skills),
        "notebook_skill_ids": [s["skill_id"] for s in notebook_skills],
    }


# Run the review (in dry-run mode by default)
print("Reviewing your custom skills workspace...")
cleanup_summary = review_and_cleanup_skills(client, dry_run=True)

# Store skill IDs for potential cleanup
if cleanup_summary["notebook_skill_ids"]:
    skills_to_cleanup = cleanup_summary["notebook_skill_ids"]
    print(f"\n💡 Tip: {len(skills_to_cleanup)} skill(s) can be cleaned up when you're done testing")

# UNCOMMENT THE LINE BELOW TO ACTUALLY DELETE THE NOTEBOOK SKILLS:
# review_and_cleanup_skills(client, dry_run=False)

## 7. 모범 사례와 프로덕션 팁 {#best-practices}

### 스킬 설계 원칙

1. **단일 책임**: 스킬 하나는 한 영역의 전문성에 집중해야 합니다
2. **명확한 문서화**: SKILL.md는 충실하면서도 간결해야 합니다
3. **오류 처리**: 스크립트는 경계 사례를 무난하게 처리해야 합니다
4. **버전 관리**: Git으로 스킬 변경을 추적하세요
5. **테스트**: 프로덕션 배포 전에 항상 스킬을 시험하세요

### 디렉터리 구조 모범 사례

```
custom_skills/
├── financial_analyzer/       # Single purpose, clear naming
│   ├── SKILL.md             # Under 5,000 tokens
│   ├── scripts/             # Modular Python/JS files
│   └── tests/               # Unit tests for scripts
├── brand_guidelines/         # Organizational standards
│   ├── SKILL.md
│   ├── REFERENCE.md         # Additional documentation
│   └── assets/              # Logos, templates
```

### 성능 최적화

| 전략 | 효과 | 구현 |
|----------|--------|----------------|
| **최소 프런트매터** | 스킬 발견이 빨라짐 | name: 64자, description: 1024자 |
| **지연 적재** | 토큰 사용 감소 | 필요할 때만 파일 참조 |
| **스킬 조합** | 중복 방지 | 거대 스킬 대신 스킬 결합 |
| **캐싱** | 응답 속도 향상 | 스킬 컨테이너 재사용 |

### 보안 고려 사항

- **API 키**: 스킬에 자격 증명을 하드코딩하지 마세요
- **데이터 프라이버시**: 스킬 파일에 민감한 데이터를 넣지 마세요
- **접근 제어**: 스킬은 워크스페이스 단위로 관리됩니다
- **검증**: 스크립트에서 입력을 정제하세요
- **감사 기록**: 컴플라이언스를 위해 스킬 사용을 기록하세요

## 다음 단계

🎉 **축하합니다!** Claude를 위한 커스텀 스킬을 만들고, 배포하고, 관리하는 방법을 배웠습니다.

### 배운 것

- ✅ 커스텀 스킬의 구조와 요구 사항
- ✅ SKILL.md와 Python 스크립트로 스킬 만들기
- ✅ API로 스킬 업로드하기
- ✅ 커스텀 스킬과 Anthropic 스킬 결합하기
- ✅ 프로덕션 배포를 위한 모범 사례
- ✅ 흔한 문제 해결하기

### 여정 이어 가기

1. **실험하기**: 예제 스킬을 여러분의 사용 사례에 맞게 수정해 보세요
2. **만들기**: 조직의 업무 흐름을 위한 스킬을 만들어 보세요
3. **최적화하기**: 토큰 사용량과 성능을 모니터링하세요
4. **공유하기**: 팀 협업을 위해 스킬을 문서화하세요

### 참고 자료

- [Claude API 문서](https://docs.anthropic.com/en/api/messages)
- [스킬 문서](https://docs.claude.com/en/docs/agents-and-tools/agent-skills/overview)
- [모범 사례](https://docs.claude.com/en/docs/agents-and-tools/agent-skills/best-practices)
- [Files API 문서](https://docs.claude.com/en/api/files-content)
- 예제 스킬 저장소(준비 중)

### 시도해 볼 만한 스킬 아이디어

- 📊 **데이터 파이프라인**: 검증이 포함된 ETL 워크플로
- 📝 **문서 템플릿**: 계약서, 제안서, 보고서
- 🔍 **코드 리뷰**: 스타일 가이드와 모범 사례
- 📈 **분석 대시보드**: KPI 추적과 시각화
- 🤖 **자동화 스위트**: 반복 작업 워크플로

즐거운 스킬 제작 되세요! 🚀